# Fake News Detection System

## Model Training

### Objective

Train a DistilBERT model for binary fake news classification.

### Tasks

- Load processed dataset
- Split train and test sets
- Tokenize statements
- Create PyTorch datasets
- Create DataLoaders
- Train DistilBERT
- Evaluate performance
- Save trained model

In [3]:
import pandas as pd
import torch

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW

from sklearn.model_selection import train_test_split

from torch.utils.data import DataLoader

import sys
import os

sys.path.append(os.path.abspath(".."))

from src.dataset import NewsDataset

In [5]:
df = pd.read_csv(
    "../data/processed/liar_binary.csv"
)

df.head()

,statement,label
0,Says the Annies List political group supports ...,0
1,When did the decline of coal start? It started...,1
2,"Hillary Clinton agrees with John McCain ""by vo...",1
3,Health care reform legislation is likely to ma...,0
4,The economic turnaround started at the end of ...,1


In [6]:
df.shape

(12765, 2)

In [7]:
tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [8]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [9]:
print(train_df["label"].value_counts())

print()

print(test_df["label"].value_counts())

label
1    5698
0    4514
Name: count, dtype: int64

label
1    1425
0    1128
Name: count, dtype: int64


In [10]:
train_dataset = NewsDataset(
    train_df,
    tokenizer
)

test_dataset = NewsDataset(
    test_df,
    tokenizer
)

In [11]:
sample = train_dataset[0]

print(sample.keys())

dict_keys(['input_ids', 'attention_mask', 'label'])


In [12]:
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["label"])

torch.Size([128])
torch.Size([128])
tensor(1)


In [13]:
BATCH_SIZE = 16

In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [15]:
batch = next(iter(train_loader))

print(batch.keys())

dict_keys(['input_ids', 'attention_mask', 'label'])


In [16]:
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["label"].shape)

torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16])


In [17]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [18]:
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(),lr=LEARNING_RATE)

In [19]:
EPOCHS = 2

In [20]:
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

In [21]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [22]:
print(model.config.num_labels)

2


In [23]:
def train_epoch(model, train_loader, optimizer, scheduler, device):

    model.train()

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [24]:
def evaluate_model(model, test_loader, device):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            predictions = torch.argmax(outputs.logits, dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [25]:
for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device
    )

    accuracy = evaluate_model(
        model,
        test_loader,
        device
    )

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.4f}")


Epoch 1/2


KeyboardInterrupt: 